# Multiagent Pattern - Program Promotion Warm-Up

This warm-up notebook introduces the Multiagent Pattern in a simple non-forensics setting before you move into the Lab 5 case. Instead of asking one agent to do everything, we give different agents different roles and let their outputs build on one another.

## How This Guided Exercise Works

The notebook controls the teaching sequence:

1. **Shared brief:** All three agents receive the same promotion question and program facts.
2. **Manual collaboration:** You define, connect, and run the agents one at a time so the handoffs are visible.
3. **Packaged collaboration:** You rebuild the same team with `Crew`, which keeps the dependency order organized.

**Purpose:** Compare the manual and `Crew` runs to see that `Crew` changes how the team is organized, not what each role is responsible for.

If you are new to this course sequence, it may help to review the earlier labs first:
- [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
- [Tool Use Pattern](../lab2_tool_use_pattern/03_lab_notebook.ipynb)
- [ReAct Pattern](../lab3_react_pattern/03b_lab_notebook.ipynb)
- [Planning Pattern](../lab4_planning_pattern/03a_planning_lab_notebook.ipynb)


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f686c08ca-989b-4083-9128-e6bc2a8c07b5_716x526-3.gif" alt="General Multiagent Pattern" width="500"/>

This figure shows the general Multiagent Pattern: different agents handle different parts of the work, and their outputs are combined into one stronger result.

## Learning Goal

By the end of this warm-up, you should be able to explain:
1. why one agent can focus on one role instead of doing everything,
2. how context is passed from one agent to another,
3. and how `Crew` packages the same collaboration workflow in a more organized way.

## What You Will Do
1. Set up the notebook and load the Lab 5 model settings.
2. Read a simple promotion question about the BS Cyber Forensics program at the University of Baltimore.
3. Define three role-specialized agents one at a time.
4. Connect the agents so downstream agents receive upstream context automatically.
5. Run the three agents step by step and inspect how context is passed.
6. Run the same workflow again with `Crew`, which keeps the dependency order organized for you.


## Quick Vocabulary

A few plain-language words before you begin:
- `Agent`: one role-specific helper that works on one part of the problem.
- `Context`: the earlier output that is passed to another agent as working notes.
- `Dependency`: the rule that one agent should wait for another agent's output first.
- `Crew`: the object that organizes a team of agents and runs them in dependency order.

This notebook shows two equally important views of the same workflow: first the collaboration is shown manually, agent by agent; then the same collaboration is shown again using `Crew`.


## Warm-Up Question

Use the following non-forensics question in this notebook:

**How should the University of Baltimore promote its BS Cyber Forensics program over the next three years so it reaches the right students, explains the program's value clearly, and uses realistic outreach methods?**

The three agents in this warm-up have different jobs:
- `AudienceAgent`: decide who the program should target first.
- `ProgramValueAgent`: decide which program strengths matter most.
- `OutreachAgent`: combine those earlier ideas into one three-year promotion plan.

The main learning goal is not marketing. The goal is to see how multiagent collaboration works when each agent has a clear role.


## Expected Agent Outputs

Keep this checklist in mind as you move through the notebook:
- `AudienceAgent` should produce: priority student groups, why each group fits, and audience advice for the team.
- `ProgramValueAgent` should produce: most important program strengths, why students should care, and message themes.
- `OutreachAgent` should produce: a final working three-year promotion plan with `Year 1`, `Year 2`, and `Year 3` outreach focus.

So in this warm-up, the first two agents prepare specialized inputs, and `OutreachAgent` writes the combined final plan.


## Notebook Structure

- **Shared setup (Steps 1-2):** Load the Lab 5 settings and create the promotion brief used by every agent.
- **Part A — Manual Collaboration (Steps 3-11):** Define the three roles, inspect the context passed between them, and run them one at a time.
- **Part B — `Crew` Collaboration (Steps 1-4):** Rebuild the same team, view its dependency graph, and run it through `Crew`.

Run the shared setup once. Part B starts with fresh agents so its results do not reuse the context accumulated during Part A.


## Part A — Build and Run the Collaboration Manually

In Part A, you will set up the notebook, define the three agents, connect their dependencies, and then run them one at a time. This keeps the collaboration visible, so you can see exactly what each agent contributes before the packaged `Crew` version appears.


### Step 1: Set Up the Notebook

This step prepares Python to create and run the promotion-team agents.

- **Inputs:** the Lab 5 folder, its `.env` file, and the repository's `src/` directory containing the `Agent` and `Crew` classes.
- **Processing:** load the model settings and import the `Agent` and `Crew` classes.
- **Output:** a ready notebook environment for the later agent steps.


In [ ]:
# Purpose: This cell supports Step 1: Set Up the Notebook by loading the libraries, settings, and data needed for this section.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

LAB_NAME = 'lab5_multiagent_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

# Add the local src/ folder so Python can import the course code used in this notebook.
src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

# Import the course-specific classes only after src/ has been added to the Python path.
from agentic_patterns.multiagent_pattern.agent import Agent
from agentic_patterns.multiagent_pattern.crew import Crew

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Model:', MODEL)


### Step 2: Create the Shared Promotion Brief

This step gives every agent the same promotion question and program facts.

- **Inputs:** the promotion question and the program facts.
- **Processing:** store them in one shared brief that every agent can use.
- **Output:** a common starting point, so later differences come from the agents' roles rather than inconsistent facts.


In [ ]:
# Purpose: This cell supports Step 2: Create the Shared Promotion Brief by preparing the question, instructions, or context used by the next model step.
PROMOTION_QUESTION = (
    'Create a three-year plan to promote the University of Baltimore BS Cyber Forensics program to prospective students. '
    'Identify the best student audiences, explain the program strengths that matter most, '
    'and recommend realistic outreach methods.'
)

# This shared brief keeps the example grounded and gives every agent the same basic facts.
PROGRAM_BRIEF = """
Program facts:
- The University of Baltimore BS Cyber Forensics program emphasizes hands-on labs, digital evidence analysis, mobile device analysis, and incident response.
- Graduates may pursue careers in digital forensics, cybersecurity operations, and investigative support roles.
- The University of Baltimore wants to attract high school seniors, community-college transfer students, and adult learners changing careers.
- The promotion budget is moderate, so the plan should favor realistic actions rather than expensive campaigns.
- The plan should cover three years, not just one semester.
""".strip()

print(PROMOTION_QUESTION)
print()
print(PROGRAM_BRIEF)


### Step 3: Define `AudienceAgent`

This step creates the first specialist, which identifies the audiences the promotion plan should prioritize.

- **Inputs:** the shared promotion brief and instructions to focus only on target audiences.
- **Processing:** configure one agent with a narrow audience-selection role.
- **Output:** `AudienceAgent`, ready to identify priority student groups without attempting the whole promotion plan.


In [ ]:
# Purpose: This cell supports Step 3: Define `AudienceAgent` by preparing the question, instructions, or context used by the next model step.
# Create the first specialist. It focuses only on target audiences.
audience_agent = Agent(
    name='AudienceAgent',
    backstory=(
        'You identify the best student audiences for academic program promotion. '
        'You focus on who the program should reach first and why.'
    ),
    # task_description = the question this one agent should answer.
    task_description=(
        f'{PROMOTION_QUESTION}\n\n'
        f'{PROGRAM_BRIEF}\n\n'
        'Focus only on the audience question. Identify the 2 or 3 target groups that matter most '
        'for this program and explain why each group is a strong fit.'
    ),
    # task_expected_output = the format we want the agent to follow.
    task_expected_output=(
        'Use these labels:\n'
        'Priority student groups:\n'
        'Why each group fits:\n'
        'Audience advice for the rest of the team:'
    ),
    llm=MODEL,
)

print(audience_agent)


### Step 4: Define `ProgramValueAgent`

This step creates a second specialist to identify the program strengths worth emphasizing.

- **Inputs:** the shared promotion brief and instructions to focus on program value.
- **Processing:** configure a second agent to interpret which program strengths matter to the selected audiences.
- **Output:** `ProgramValueAgent`, ready to add message themes rather than repeat the audience analysis.


In [ ]:
# Purpose: This cell supports Step 4: Define `ProgramValueAgent` by preparing the question, instructions, or context used by the next model step.
# Create the second specialist. It focuses on the value of the program.
program_value_agent = Agent(
    name='ProgramValueAgent',
    backstory=(
        'You explain the strongest value of an academic program in student-friendly language. '
        'You focus on program strengths, benefits, and message themes.'
    ),
    task_description=(
        f'{PROMOTION_QUESTION}\n\n'
        f'{PROGRAM_BRIEF}\n\n'
        'Focus only on program value. Explain which strengths of the University of Baltimore BS Cyber Forensics program '
        'should be highlighted most clearly to prospective students.'
    ),
    task_expected_output=(
        'Use these labels:\n'
        'Most important program strengths:\n'
        'Why students should care:\n'
        'Message themes for outreach materials:'
    ),
    llm=MODEL,
)

print(program_value_agent)


### Step 5: Define `OutreachAgent`

This step creates the final specialist, which turns the team's findings into a promotion plan.

- **Inputs:** the shared promotion brief and instructions to create a three-year outreach plan.
- **Processing:** configure a third agent to synthesize the earlier specialists' findings.
- **Output:** `OutreachAgent`, ready to turn the team inputs into one staged promotion plan.


In [ ]:
# Purpose: This cell supports Step 5: Define `OutreachAgent` by preparing the question, instructions, or context used by the next model step.
# Create the final specialist. It turns earlier ideas into one phased outreach plan.
outreach_agent = Agent(
    name='OutreachAgent',
    backstory=(
        'You create realistic outreach plans for academic programs. '
        'You combine earlier audience and value insights into one practical phased plan.'
    ),
    task_description=(
        f'{PROMOTION_QUESTION}\n\n'
        f'{PROGRAM_BRIEF}\n\n'
        'Use the earlier agent context to create one practical three-year outreach plan. '
        'Include a simple Year 1, Year 2, and Year 3 timeline.'
    ),
    task_expected_output=(
        'Use these labels:\n'
        'Priority audience:\n'
        'Main messages:\n'
        'Year 1 outreach focus:\n'
        'Year 2 outreach focus:\n'
        'Year 3 outreach focus:\n'
        'Final working promotion plan:'
    ),
    llm=MODEL,
)

print(outreach_agent)


### Step 6: Connect the Agents

This step defines how each specialist's findings move to the next role.

- **Inputs:** the three configured agents.
- **Processing:** use `>>` to create dependency links, which tell the notebook which earlier output becomes context for a later agent.
- **Output:** a visible collaboration graph: `AudienceAgent` passes its findings to both later agents, and `ProgramValueAgent` also passes its findings to `OutreachAgent`.


In [ ]:
# Purpose: This cell supports Step 6: Connect the Agents by preparing or examining the evidence used in this section.
# `>>` means: pass this agent's output to the next agent as context.
audience_agent >> program_value_agent
audience_agent >> outreach_agent
program_value_agent >> outreach_agent

# Show the dependency structure after the arrows are added.
print('AudienceAgent dependents:', audience_agent.dependents)
print('ProgramValueAgent dependencies:', program_value_agent.dependencies)
print('OutreachAgent dependencies:', outreach_agent.dependencies)

print('Part A setup is complete.')


### Step 7: Run `AudienceAgent`

This step produces the team's first specialist finding: an audience strategy.

- **Inputs:** the shared promotion brief and `AudienceAgent`'s audience-selection instructions.
- **Processing:** the agent reviews the brief and selects priority student groups.
- **Output:** an audience strategy that becomes context for the next agent. Look for priority groups and a reason each group fits.


In [ ]:
# Purpose: This cell supports Step 7: Run `AudienceAgent` by running the model or agent action and saving its result for review.
audience_output = audience_agent.run()
display(Markdown('### AudienceAgent Output\n\n' + audience_output))


`AudienceAgent` has now contributed the first useful layer of the plan: who the program should reach first. The downstream agents can use this as working guidance instead of starting from scratch.


### Step 8: Inspect the Context Passed to `ProgramValueAgent`

This step makes the first agent-to-agent handoff visible.

- **Inputs:** the audience strategy produced in Step 7.
- **Processing:** no new model call occurs; the cell displays the stored context that will be supplied to `ProgramValueAgent`.
- **Output:** visible evidence of the first handoff. Check that the next agent receives the audience findings, not a blank prompt.


In [ ]:
# Purpose: This cell supports Step 8: Inspect the Context Passed to `ProgramValueAgent` by showing the result in a form that is easier to review.
display(Markdown('### ProgramValueAgent Received Context\n\n```text\n' + program_value_agent.context + '\n```'))


### Step 9: Run `ProgramValueAgent`

This step adds program-value recommendations to the earlier audience strategy.

- **Inputs:** the shared promotion brief, `ProgramValueAgent`'s role instructions, and the audience strategy passed from Step 7.
- **Processing:** the agent uses the audience findings to select relevant program strengths and message themes.
- **Output:** value-focused recommendations that extend the audience analysis instead of repeating it.


In [ ]:
# Purpose: This cell supports Step 9: Run `ProgramValueAgent` by running the model or agent action and saving its result for review.
program_value_output = program_value_agent.run()
display(Markdown('### ProgramValueAgent Output\n\n' + program_value_output))


`ProgramValueAgent` adds the second layer of the plan: what to say and why students should care. At this point, the team has both audience guidance and message guidance.


### Step 10: Inspect the Context Passed to `OutreachAgent`

This step shows the complete set of specialist findings before synthesis.

- **Inputs:** the audience strategy and the value-focused recommendations.
- **Processing:** no new model call occurs; the cell displays the combined context that will be supplied to `OutreachAgent`.
- **Output:** visible evidence that the final agent receives both specialist findings before it writes the plan.


In [ ]:
# Purpose: This cell supports Step 10: Inspect the Context Passed to `OutreachAgent` by showing the result in a form that is easier to review.
display(Markdown('### OutreachAgent Received Context\n\n```text\n' + outreach_agent.context + '\n```'))


### Step 11: Run `OutreachAgent`

This step synthesizes the two specialist findings into one staged promotion plan.

- **Inputs:** the shared promotion brief, `OutreachAgent`'s synthesis instructions, and both earlier agent outputs.
- **Processing:** the agent combines the audience and value findings into a staged outreach strategy.
- **Output:** one three-year promotion plan. Check that it uses the earlier findings and distinguishes `Year 1`, `Year 2`, and `Year 3`.


In [ ]:
# Purpose: This cell supports Step 11: Run `OutreachAgent` by running the model or agent action and saving its result for review.
outreach_output = outreach_agent.run()
display(Markdown('### OutreachAgent Final Promotion Plan\n\n' + outreach_output))


`OutreachAgent` has now turned both earlier outputs into the final working plan. This is the main Multiagent Pattern idea: different roles contribute different pieces, and the final agent combines them into one stronger answer.


## Part B — Run the Same Workflow with `Crew`

`Crew` does not replace the agents. It organizes the same team and runs them in dependency order. This means Part B is not a different workflow. It is the same collaboration pattern packaged in a cleaner way.


### Step 1: Prepare a Fresh Team for `Crew`

This step creates fresh setup code so the `Crew` run does not reuse Part A's context.

- **Inputs:** the shared promotion brief and the three role definitions from Part A.
- **Processing:** define a helper that rebuilds fresh agents and a display helper for their outputs.
- **Output:** reusable setup code that prevents Part B from reusing context accumulated during the manual run.


In [ ]:
# Purpose: This cell supports Step 1: Prepare a Fresh Team for `Crew` by defining reusable helper code that performs the work described here.
def build_program_promotion_agents():
    """Create a fresh copy of the same three-agent team for Part B."""
    audience_agent = Agent(
        name='AudienceAgent',
        backstory=(
            'You identify the best student audiences for academic program promotion. '
            'You focus on who the program should reach first and why.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Focus only on the audience question. Identify the 2 or 3 target groups that matter most '
            'for this program and explain why each group is a strong fit.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Priority student groups:\n'
            'Why each group fits:\n'
            'Audience advice for the rest of the team:'
        ),
        llm=MODEL,
    )

    program_value_agent = Agent(
        name='ProgramValueAgent',
        backstory=(
            'You explain the strongest value of an academic program in student-friendly language. '
            'You focus on program strengths, benefits, and message themes.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Focus only on program value. Explain which strengths of the University of Baltimore BS Cyber Forensics program '
            'should be highlighted most clearly to prospective students.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Most important program strengths:\n'
            'Why students should care:\n'
            'Message themes for outreach materials:'
        ),
        llm=MODEL,
    )

    outreach_agent = Agent(
        name='OutreachAgent',
        backstory=(
            'You create realistic outreach plans for academic programs. '
            'You combine earlier audience and value insights into one practical phased plan.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Use the earlier agent context to create one practical three-year outreach plan. '
            'Include a simple Year 1, Year 2, and Year 3 timeline.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Priority audience:\n'
            'Main messages:\n'
            'Year 1 outreach focus:\n'
            'Year 2 outreach focus:\n'
            'Year 3 outreach focus:\n'
            'Final working promotion plan:'
        ),
        llm=MODEL,
    )

    # Recreate the same dependency graph for the Crew part.
    audience_agent >> program_value_agent
    audience_agent >> outreach_agent
    program_value_agent >> outreach_agent

    return audience_agent, program_value_agent, outreach_agent


def run_crew_with_markdown(crew: Crew):
    """Run the crew in dependency order and show each output as readable Markdown."""
    outputs = {}
    for agent in crew.topological_sort():
        output = agent.run()
        outputs[agent.name] = output
        display(Markdown(f'### {agent.name} Output\n\n{output}'))
    return outputs


### Step 2: Build the Same Team Inside `Crew`

This step gives `Crew` the same agents and handoffs used in the manual workflow.

- **Inputs:** the fresh-team helper from Step 1.
- **Processing:** create the same three agents inside a `Crew` context, which records their dependency graph.
- **Output:** a fresh `Crew` with the same roles and handoffs used in Part A.


In [ ]:
# Purpose: This cell supports Step 2: Build the Same Team Inside `Crew` by creating and connecting the agents that will complete this workflow.
with Crew() as crew:
    crew_audience_agent, crew_program_value_agent, crew_outreach_agent = build_program_promotion_agents()


### Step 3: Visualize the Crew Graph

This step turns the stored dependency rules into a visual collaboration map.

- **Inputs:** the `Crew` dependency graph created in Step 2.
- **Processing:** render the graph; no agent analyzes the promotion brief in this step.
- **Output:** a visual check of the collaboration order.

When you inspect the graph, look for this pattern:
- `AudienceAgent` feeds both later agents.
- `ProgramValueAgent` also feeds `OutreachAgent`.
- `OutreachAgent` runs last because it depends on both earlier outputs.


In [ ]:
# Purpose: This cell supports Step 3: Visualize the Crew Graph by preparing or examining the evidence used in this section.
crew.plot()


### Step 4: Run the Full Crew

This step runs the complete collaboration through one coordinated `Crew` call.

- **Inputs:** the fresh `Crew`, its dependency graph, and the shared promotion brief.
- **Processing:** `Crew` runs each agent in dependency order and passes the earlier outputs as context.
- **Output:** the same sequence of role-specific outputs as Part A, produced through one coordinated call. Compare the reasoning with the manual run rather than treating it as a different workflow.


In [ ]:
# Purpose: This cell supports Step 4: Run the Full Crew by preparing or examining the evidence used in this section.
crew_outputs = run_crew_with_markdown(crew)


## Why This Warm-Up Matters

This notebook is a non-forensics example, but the same role sequence appears in the main Lab 5 case:
- `InvestigationAgent` organizes the technical timeline and identifies the transmission claim to test.
- `EvidenceVerificationAgent` checks that claim against the files and logs.
- `CustodyAuditAgent` reviews the evidence-handling record and explains how it affects confidence.
- You compare the role outputs and write the final evidence-bounded conclusion.

Next, move to [03b_multiagent_forensic_workflow.ipynb](./03b_multiagent_forensic_workflow.ipynb) for the full forensic multiagent case.
